# 02장. 급식 데이터 정리와 그래프

| 핵심 질문 | 학습 시간 |
|---|---:|
| 복잡한 메뉴 글자를 어떻게 표로 바꿀까? | 2회차 · 약 180분 |


## 이 장에서 배울 내용

- HTML 줄바꿈과 알레르기 번호를 메뉴명에서 분리할 수 있다.
- 열량과 영양 문자열을 숫자 열로 바꾸는 이유를 설명할 수 있다.
- 표와 막대그래프에서 상대적인 차이를 읽을 수 있다.


## 생각 열기

NEIS에서 받은 메뉴에는 `<br/>`, 괄호 속 번호, `kcal` 같은 표시가 함께 들어 있습니다. 사람이 보기에는 뜻이 분명하지만 그대로는 계산하기 어렵습니다. 분석에 필요한 부분을 나누고 숫자를 꺼내는 과정이 필요합니다.


## 핵심 용어

| 용어 | 뜻 |
|---|---|
| **전처리** | 분석 전에 데이터를 정리하고 변환하는 과정 |
| **결측값** | 기록되지 않아 비어 있는 값 |
| **DataFrame** | 행과 열로 이루어진 Pandas 표 |
| **시각화** | 숫자의 관계를 그래프로 표현하는 일 |


## 개념 익히기


전처리는 요리 전 재료 손질과 같습니다. 메뉴 문자열에는 줄바꿈 표시, 괄호 속 알레르기 번호, 단위가 붙은 숫자가 함께 있습니다. 사람은 눈으로 구분하지만 Python은 규칙을 알려 주어야 합니다.

숫자가 비어 있다고 0이라고 단정하면 안 됩니다. ‘기록 없음’과 ‘영양소 0’은 다른 뜻이므로 결측값은 데이터 부족으로 다룹니다.


## 활동 전 생각


‘미트볼파스타 (1.2.5.6)&lt;br/&gt;피클’을 종이에 써서 메뉴명, 번호, 줄바꿈 표시를 서로 다른 색으로 표시해 보세요.


## 예상하기

- 메뉴 원문에서 HTML 표시가 사라진다.
- 날짜별 열량 막대가 5개 나타난다.


## 활동 1. 한 행을 손질하기


### 코드 살펴보기


1. `sample_menu`에는 첫 급식의 메뉴 원문을 저장합니다.<br>
2. `split_dishes`는 줄바꿈 표시를 기준으로 메뉴를 나눕니다.<br>
3. `extract_allergy_codes`와 `parse_calories`는 번호와 열량을 각각 숫자로 바꿉니다.


In [ ]:
import sys
from pathlib import Path

current_folder = Path.cwd().resolve()
for candidate in (current_folder, *current_folder.parents):
    if (candidate / "jupyter_course" / "notebook_support.py").is_file():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError(
        "프로젝트 폴더를 찾지 못했습니다. neis-meal-ai 폴더에서 "
        r".\.venv\Scripts\python.exe -m notebook 명령으로 다시 시작하세요."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from jupyter_course.notebook_support import course_setup

setup = course_setup(PROJECT_ROOT)
PROJECT_ROOT = setup["root"]
raw_rows = setup["rows"]
meal_df = setup["frame"]
data_source = setup["source"]
print("프로젝트 폴더:", PROJECT_ROOT)
print("데이터 출처:", data_source)
print("급식 행 수:", len(raw_rows))

from neis_meal_ai.cleaning import (
    extract_allergy_codes,
    parse_calories,
    split_dishes,
)

sample_menu = raw_rows[0]["DDISH_NM"]
print("원문 일부:", sample_menu[:120])
print("메뉴 목록:", split_dishes(sample_menu))
print("알레르기 번호:", extract_allergy_codes(sample_menu))
print("열량 숫자:", parse_calories(raw_rows[0]["CAL_INFO"]))


### 결과 해석하기

메뉴 이름은 읽기 쉬운 목록이 되고, 알레르기 번호는 별도 튜플로 보존됩니다. 삭제가 아니라 분리입니다.


## 활동 2. 정제 표와 결측값 확인


### 코드 살펴보기


1. `learning_columns`는 학습에 사용할 열의 순서를 정합니다.<br>
2. `meal_df[learning_columns]`는 선택한 열만 같은 순서로 보여 줍니다.<br>
3. `isna().sum()`은 열마다 비어 있는 값의 개수를 셉니다.


In [ ]:
learning_columns = [
    "date", "menu_text", "calories", "carbs_g",
    "protein_g", "fat_g", "dish_count",
]
print(meal_df[learning_columns].to_string(index=False))
print()
print("열별 빈 값 개수:")
print(meal_df[learning_columns].isna().sum().to_string())


### 결과 해석하기

한 행은 하루, 한 열은 같은 종류의 특징입니다. 빈 값 개수가 0이면 이번 예비 자료의 영양 수치가 모두 기록된 것입니다.


## 활동 3. 날짜별 열량 막대그래프


### 코드 살펴보기


1. `meal_df.plot.bar`는 `date`를 가로축, `calories`를 세로축으로 그립니다.<br>
2. `set_title`, `set_xlabel`, `set_ylabel`은 그래프의 제목과 축 이름을 붙입니다.<br>
3. `NEIS_JUPYTER_VERIFY`가 설정된 자동 검증에서는 화면 대신 그래프를 닫습니다.


In [ ]:
import os
import matplotlib
if os.getenv("NEIS_JUPYTER_VERIFY") == "1":
    matplotlib.use("Agg")
import matplotlib.pyplot as plt

ax = meal_df.plot.bar(x="date", y="calories", legend=False, color="#4C78A8")
ax.set_title("Namak High School lunch calories (relative view)")
ax.set_xlabel("date")
ax.set_ylabel("kcal")
plt.tight_layout()
if os.getenv("NEIS_JUPYTER_VERIFY") != "1":
    plt.show()
else:
    plt.close()

chapter_result = {
    "chapter": "02",
    "clean_rows": len(meal_df),
    "columns": list(meal_df.columns),
    "chart_ready": True,
}


### 결과 해석하기

막대의 높이는 날짜 사이의 상대 차이를 보여 줍니다. 이 그래프만으로 건강함이나 학생에게 맞는 식단을 판단할 수 없습니다.


## 탐구 활동

graph_column을 protein_g 또는 carbs_g로 바꾸어 가장 높은 날짜가 달라지는지 확인하세요.

먼저 기본값으로 실행한 뒤 한 곳만 바꾸어 결과를 비교합니다.


In [ ]:
graph_column = "protein_g"
highest_row = meal_df.loc[meal_df[graph_column].idxmax()]
print(graph_column, "값이 가장 큰 날짜:", highest_row["date"])
print("값:", highest_row[graph_column])


### 관찰 기록

- 바꾼 것:  
- 달라진 결과:  
- 그렇게 된 까닭:


## 확인 문제

1. 알레르기 번호를 메뉴명에서 지우기만 하지 않고 별도 열에 보존하는 이유는 무엇인가요?
2. 결측값과 숫자 0은 왜 다른가요?
3. 열량 그래프를 건강 순위라고 부르면 안 되는 이유는 무엇인가요?


## 정답과 해설


1. 표시와 분석에 다시 사용해야 하는 정보이기 때문입니다.<br>
2. 결측값은 기록이 없다는 뜻이고 0은 실제 값이 0이라는 뜻입니다.<br>
3. 열량 한 가지 수치만으로 개인의 건강 적합성을 판단할 수 없기 때문입니다.


## 핵심 정리

- 전처리는 원본을 메뉴 목록, 번호, 숫자 열로 나눈다.
- 결측값을 0으로 단정하지 않는다.
- 그래프는 상대 차이를 관찰하는 도구이지 건강 판정표가 아니다.

### 다음 장에서 배울 내용

03장에서는 메뉴 글자를 숫자 벡터로 바꾸는 TF-IDF를 배웁니다.


In [ ]:
import json
print("__CHAPTER_RESULT__=" + json.dumps(chapter_result, ensure_ascii=False))
